In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# !pip install menpo
# !pip install fvcore
# !git clone https://github.com/im-xiaoming/xiaoying.git

In [ ]:
# !cp -r /content/drive/MyDrive/Ying/faces_extracted.zip /content
# !unzip -q /content/faces_extracted.zip -d /content/

# !cp -r /content/drive/MyDrive/VN-celeb.zip /content
# !unzip -q /content/VN-celeb.zip -d /content/

In [ ]:
import torch
import os
import torch.nn as nn
import torchvision.transforms as transforms
from xiaoying.utils import get_loader, get_model, get_head, get_optimizer, set_lr
from xiaoying.trainer import Trainer

In [ ]:
TRANSFORMS = transforms.Compose([
        # transforms.Resize((112, 112)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
    ])


DATA_PATH = '/content/VN-celeb'
CHECKPOINT = '/content/ir_18_checkpoint_2.pth'
BATCH_SIZE = 512
MODEL_NAME = 'ir_18' # ir_18, ir_50, vit
LEARNING_RATE = 0.1 # 0.01, 0.001, 0.0001
EPOCHS = 5

# Augmentation
low_res_augmentation_prob = 0.0
crop_augmentation_prob = 0.0
photometric_augmentation_prob = 0.0


# Head params
m = 0.4 # 0.3, 0.5,...
h = 0.333
s = 64
t_alpha = 0.99
CLASS_NUM = len(os.listdir(DATA_PATH))
OUTPUT_SIZE = 512 # 768, 1024...

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
criterion = nn.CrossEntropyLoss()

dataloader = get_loader(data_path=DATA_PATH, transforms=TRANSFORMS, batch_size=BATCH_SIZE, shuffle=True,
                        low_res_augmentation_prob=low_res_augmentation_prob,
                        crop_augmentation_prob=crop_augmentation_prob,
                        photometric_augmentation_prob=photometric_augmentation_prob)
model = get_model(MODEL_NAME, device)
head = get_head(device, OUTPUT_SIZE, CLASS_NUM, m, h, s, t_alpha)
optimizer = get_optimizer(model=model, model_name=MODEL_NAME, head=head, lr=LEARNING_RATE, opt_type='sgd') # opt_type: ['sgd', 'adamw']
# set_lr(optimizer, LEARNING_RATE)

In [ ]:
trainer = Trainer(dataloader, model, MODEL_NAME, head, optimizer, criterion, device, EPOCHS)
# trainer._load_checkpoint(CHECKPOINT)
trainer.train()